<hr style="border: 6px solid#003262;" />

<div align="center">
    <img src="images/spans_agent.png" align="center" width="20%">
</div>

<br>

# TRACING AGENTS USING EDD

<br>

**About:** Add OpenTelemetry tracing to the multi-tool agent from notebook 01 using Arize Phoenix, making every LLM call and tool invocation observable as structured spans.

**Learning Goals:** (1) Set up Arize Phoenix and register an OpenTelemetry tracer provider. (2) Instrument agent tools and router functions with span types (tool, chain, agent). (3) Run the agent and observe the resulting trace structure in Phoenix. (4) Understand how tracing enables EDD by turning agent behavior into queryable data.

**Keywords:** opentelemetry, tracing, spans, arize phoenix, agent observability, edd

**Prerequisite Knowledge:** (1) `01_evaluating_agents.ipynb` - agent tools, router loop, and EDD concept, (2) Python decorators

**Target User:** Developers who have built a basic LLM agent (per notebook 01) and want to make its behavior observable without manually inserting print statements.

<hr style="border: 4px solid#003262;" />

<a name='Part_table_contents' id="Part_table_contents"></a>

#### CONTENTS

> #### [PART 0: SETUP](#Part_0)
> #### [PART 1: PHOENIX AND OPENTELEMETRY](#Part_1)
> #### [PART 2: TRACING THE AGENT](#Part_2)
> #### [PART 3: RUNNING A TRACED QUERY](#Part_3)
> #### [PART 4: EVALUATION-DRIVEN IMPROVEMENT](#Part_4)

<br>

<a id='Part_0'></a>

<hr style="border: 2px solid#003262;" />

#### PART 0

## **SETUP** AND ENVIRONMENT

The agent from notebook 01 works, but its inner life is opaque. When it returns a wrong answer, all you see is the final string. You cannot tell whether the router picked the wrong tool, the SQL came back malformed, the analysis LLM misread the data, or the visualization tool returned code that never runs. Print statements can peek at one call at a time - they do not scale across a batch of queries, and they do not let you slice results by tool or by span type.

**Tracing is what turns the agent into a queryable object.** Every LLM call, every tool invocation, every intermediate step becomes a structured span with typed attributes, a duration, and a parent span it belongs to. A batch of runs becomes a tree of spans you can query, filter, and score - which is the raw material for the evaluations in notebook 03.

This notebook wires the agent into **Arize Phoenix**, an open-source LLM observability backend, using the **OpenInference** semantic conventions on top of **OpenTelemetry**. The agent logic from notebook 01 is unchanged. Only three things are added: a tracer registration, function decorators that label span kinds, and a manually created root span for the whole agent run.

<div align="center" style="font-size:12px; font-family:FreeMono; font-weight: 100; font-stretch:ultra-condensed; line-height: 1.0; color:#2A2C2B">
    <img src="images/edd_loop.png" align="center" width="55%" padding="10"><br>
    <br>
    This notebook implements the "Instrument" and "Collect" stages of the cycle. Notebook 03 adds "Evaluate" and "Improve".
</div>

**How this notebook is organized**

- **Part 1** sets up Phoenix, registers a tracer provider, and turns on auto-instrumentation for the OpenAI client.
- **Part 2** re-declares the agent tools and router with span-kind decorators (`@tracer.tool()`, `@tracer.chain()`) so each function becomes a labeled span.
- **Part 3** runs a query through the fully traced agent and shows what to look for in the Phoenix UI.
- **Part 4** connects tracing to action: a trace is only valuable if it changes what you do next.

___

**Note:** Phoenix must be running locally before executing this notebook. Start it with `phoenix serve` in a separate terminal (the older `python -m phoenix.server.main serve` command still works). The default collector endpoint is `http://localhost:6006/v1/traces`, and the UI is at `http://localhost:6006`.

___

In [ ]:
from openai import OpenAI
import pandas as pd
import os
import json
import duckdb
import pydantic
from pydantic import BaseModel, Field
from IPython.display import Markdown
from typing import Any, Dict, List, Sequence, Union
from dotenv import load_dotenv

# Phoenix + OpenInference tracing stack.
# Verified against arize-phoenix + arize-phoenix-otel + openinference-instrumentation-openai
# packages available on PyPI as of 2026-08-31. Re-check import paths at
# https://arize.com/docs/phoenix/tracing/how-to-tracing/setup-tracing/instrument
import phoenix as px
from phoenix.otel import register
from openinference.instrumentation.openai import OpenAIInstrumentor
from opentelemetry.trace import StatusCode

load_dotenv()
AI_API_KEY = os.getenv("OPENAI_API_KEY")
client = OpenAI(api_key=AI_API_KEY)

# Match notebook 01. Failure modes surface on the mini model, and the cost of
# running a small batch stays negligible. Swap to "gpt-4o" only if you want to
# see how the same failures diminish (they do not vanish) on the larger model.
MODEL = "gpt-4o-mini"
TRANSACTION_DATA_FILE_PATH = "data/Store_Sales_Price_Elasticity_Promotions_Data.parquet"

<!--Navigate back to table of contents-->
<div align="left" style="text-align: left; background-color:#003262;">
    <span>
        <hr style="border: 8px solid#003262;" />
        <a style="color:#FFFFFF; background-color:#003262; border:1px solid #FFFFFF; border-color:#FFFFFF;border-radius:0px;border-width:0px;display:inline-block;font-family:arial,helvetica,sans-serif;font-size:24px;letter-spacing:0px;line-height:20px;padding:24px 40px;text-align:left;text-decoration:none; align:left"> 
            <strong>CONCEPT</strong> CHECK 
        </a>        
    </span>
</div>
<!-------------------------------------->

> **The instrumentation stack imports three separate packages: `phoenix`, `phoenix.otel`, and `openinference.instrumentation.openai`. Name what each one is responsible for in one sentence each. Then predict which import would break first if you started Phoenix on a non-default port and forgot to update `PHOENIX_COLLECTOR_ENDPOINT`.**

<br>

```python
# Write your three one-sentence explanations and prediction as comments.
```

<hr style="border: 2px solid#003262;" />

<!--Navigate back to table of contents-->
<div alig="right" style="text-align: right">
    <span>
        <a style="color:#FFFFFF; background-color:#003262; border:1px solid #FFFFFF; border-color:#FFFFFF;border-radius:5px;border-width:0px;display:inline-block;font-family:arial,helvetica,sans-serif;font-size:10px;letter-spacing:0px;line-height:10px;padding:10px 20px;text-align:center;text-decoration:none; align:center" href="#Part_table_contents" name="Table of Contents"  id="Part_table_contents"> 
            Table of Contents 
        </a>
    </span>
</div>
<!-------------------------------------->

<a id='Part_1'></a>

<hr style="border: 2px solid#003262;" />

#### PART 1

## **PHOENIX AND OPENTELEMETRY**: Setting Up the Observer

<div align="center" style="font-size:12px; font-family:FreeMono; font-weight: 100; font-stretch:ultra-condensed; line-height: 1.0; color:#2A2C2B">
    <img src="images/agent_dark_background.png" align="center" width="30%" padding="10"><br>
    <br>
</div>

#### CONTENTS:

> [PART 1.1: TRACER PROVIDER](#Part_1_1)<br>
> [PART 1.2: AUTO-INSTRUMENTATION](#Part_1_2)<br>

<a id='Part_1_1'></a>

<hr style="border: 1px solid#003262;" />

#### PART 1.1: TRACER PROVIDER

<br>

OpenTelemetry is a vendor-neutral observability standard. Arize Phoenix is an open-source backend that collects OpenTelemetry data and provides a UI for inspecting traces.

The setup requires two steps:

1. **Register a tracer provider** - `register()` creates a `TracerProvider` configured to send span data to the Phoenix collector endpoint. The `project_name` groups related traces together in the Phoenix UI.
2. **Get a tracer** - `tracer_provider.get_tracer(__name__)` returns a tracer object. Functions decorated with `@tracer.tool()`, `@tracer.chain()`, or used inside `tracer.start_as_current_span(...)` will emit spans through this tracer.

___

**Note:** `auto_instrument=True` in `register()` enables automatic instrumentation of supported libraries. For OpenAI calls, this means every `client.chat.completions.create()` call emits an LLM-typed span automatically - without any additional code changes to the functions from notebook 01.

___

In [ ]:
PROJECT_NAME = "tracing_agent"
PHOENIX_COLLECTOR_ENDPOINT = "http://localhost:6006/v1/traces"

tracer_provider = register(
    project_name=PROJECT_NAME,
    endpoint=PHOENIX_COLLECTOR_ENDPOINT,
    auto_instrument=True
)

tracer = tracer_provider.get_tracer(__name__)

<!--Navigate back to table of contents-->
<div align="left" style="text-align: left; background-color:#003262;">
    <span>
        <hr style="border: 8px solid#003262;" />
        <a style="color:#FFFFFF; background-color:#003262; border:1px solid #FFFFFF; border-color:#FFFFFF;border-radius:0px;border-width:0px;display:inline-block;font-family:arial,helvetica,sans-serif;font-size:24px;letter-spacing:0px;line-height:20px;padding:24px 40px;text-align:left;text-decoration:none; align:left"> 
            <strong>CONCEPT</strong> CHECK 
        </a>        
    </span>
</div>
<!-------------------------------------->

> **What is the difference between a tracer provider and a tracer object? Write a one-sentence explanation of each, and explain why they are separate concepts.**

<br>

```python
# Write your explanation as comments.
```

<hr style="border: 2px solid#003262;" />

<!--Navigate back to table of contents-->
<div alig="right" style="text-align: right">
    <span>
        <a style="color:#FFFFFF; background-color:#003262; border:1px solid #FFFFFF; border-color:#FFFFFF;border-radius:5px;border-width:0px;display:inline-block;font-family:arial,helvetica,sans-serif;font-size:10px;letter-spacing:0px;line-height:10px;padding:10px 20px;text-align:center;text-decoration:none; align:center" href="#Part_table_contents" name="Table of Contents"  id="Part_table_contents"> 
            Table of Contents 
        </a>
    </span>
</div>
<!-------------------------------------->

<a id='Part_1_2'></a>

<hr style="border: 1px solid#003262;" />

#### PART 1.2: AUTO-INSTRUMENTATION

<br>

The `OpenAIInstrumentor` patches the OpenAI client so that every `chat.completions.create()` call automatically emits an LLM-typed span. This span captures:

- The full prompt sent to the model (input)
- The model's response (output)
- Token counts and model name (metadata)

Auto-instrumentation is the lowest-effort way to get visibility into LLM calls. It requires no changes to the functions written in notebook 01 - the instrumentation is applied at import time via monkey-patching.

In [ ]:
# Instrument the OpenAI client. Must be called after the tracer provider is registered.
OpenAIInstrumentor().instrument(tracer_provider=tracer_provider)

<!--Navigate back to table of contents-->
<div align="left" style="text-align: left; background-color:#003262;">
    <span>
        <hr style="border: 8px solid#003262;" />
        <a style="color:#FFFFFF; background-color:#003262; border:1px solid #FFFFFF; border-color:#FFFFFF;border-radius:0px;border-width:0px;display:inline-block;font-family:arial,helvetica,sans-serif;font-size:24px;letter-spacing:0px;line-height:20px;padding:24px 40px;text-align:left;text-decoration:none; align:left"> 
            <strong>CONCEPT</strong> CHECK 
        </a>        
    </span>
</div>
<!-------------------------------------->

> **Auto-instrumentation captures every OpenAI call automatically. Name one situation where you would want to suppress automatic tracing for a specific function call, and how you would do it using Phoenix's `suppress_tracing()` context manager.**

<br>

```python
# Write your answer as a comment. Hint: the suppress_tracing import is shown in notebook 03.
```

<hr style="border: 2px solid#003262;" />

<!--Navigate back to table of contents-->
<div alig="right" style="text-align: right">
    <span>
        <a style="color:#FFFFFF; background-color:#003262; border:1px solid #FFFFFF; border-color:#FFFFFF;border-radius:5px;border-width:0px;display:inline-block;font-family:arial,helvetica,sans-serif;font-size:10px;letter-spacing:0px;line-height:10px;padding:10px 20px;text-align:center;text-decoration:none; align:center" href="#Part_table_contents" name="Table of Contents"  id="Part_table_contents"> 
            Table of Contents 
        </a>
    </span>
</div>
<!-------------------------------------->

<a id='Part_2'></a>

<hr style="border: 2px solid#003262;" />

#### PART 2

## **TRACING THE AGENT**: Instrumented Tools and Router

<div align="center" style="font-size:12px; font-family:FreeMono; font-weight: 100; font-stretch:ultra-condensed; line-height: 1.0; color:#2A2C2B">
    <img src="images/tool1.png" align="center" width="20%" padding="10"><br>
    <br>
</div>

The agent from notebook 01 is reproduced here with three additions - Phoenix decorator span types applied to the functions:

- `@tracer.tool()` - applied to `lookup_sales_data` and `analyze_sales_data`. Tool spans represent calls to external tools. They appear in Phoenix under the "tool" span kind.
- `@tracer.chain()` - applied to `handle_tool_calls`, `extract_chart_config`, `create_chart`, and `generate_visualization`. Chain spans represent sequential steps within a pipeline - linking intermediate steps that are not external tools or final LLM calls.
- The main agent span uses `tracer.start_as_current_span()` with `openinference_span_kind="agent"`. This span wraps the entire agent run, making it the root of the trace.

Everything inside a span - including any nested function calls - becomes a child span automatically. This nesting is what makes a trace a tree rather than a flat list of events.

<strong style="color:red">KEY CONSIDERATION:</strong> Span type matters for evaluation. In notebook 03, evals query spans by kind (`span_kind == 'LLM'`, `span_kind == 'AGENT'`). If you apply the wrong decorator, your evals will miss the spans they need.

In [ ]:
SQL_GENERATION_PROMPT = """
Generate an SQL query based on the prompt that follows. Do not reply with anything besides the SQL query.
The prompt is: {prompt}

The available columns are: {columns}
The table name is: {table_name}
"""


def generate_sql_query(prompt: str, columns: list[str], table_name: str) -> str:
    """Generate a SQL query from a natural language prompt (auto-instrumented by OpenAIInstrumentor)."""
    formatted_prompt = SQL_GENERATION_PROMPT.format(
        prompt=prompt, columns=columns, table_name=table_name
    )
    response = client.chat.completions.create(
        model=MODEL,
        messages=[{"role": "user", "content": formatted_prompt}],
    )
    return response.choices[0].message.content.strip()


@tracer.tool()
def lookup_sales_data(prompt: str) -> str:
    """Query sales data using LLM-generated SQL. Emits a tool-typed span."""
    try:
        table_name = "sales"
        df = pd.read_parquet(TRANSACTION_DATA_FILE_PATH)
        duckdb.sql(f"CREATE TABLE IF NOT EXISTS {table_name} AS SELECT * FROM df")
        sql_query = generate_sql_query(prompt, df.columns.tolist(), table_name)
        sql_query = sql_query.strip().replace("```sql", "").replace("```", "")
        result = duckdb.sql(sql_query).df()
        return result.to_string()
    except Exception as e:
        return f"Error accessing data: {e}" 

In [ ]:
DATA_ANALYSIS_PROMPT = """
Analyze the following data: {data}
Your job is to answer the following question: {prompt}
"""


@tracer.tool()
def analyze_sales_data(prompt: str, data: str) -> str:
    """Analyze sales data with an LLM. Emits a tool-typed span."""
    formatted_prompt = DATA_ANALYSIS_PROMPT.format(data=data, prompt=prompt)
    response = client.chat.completions.create(
        model=MODEL,
        messages=[{"role": "user", "content": formatted_prompt}],
    )
    analysis = response.choices[0].message.content
    return analysis if analysis else "No analysis could be generated" 

In [ ]:
CHART_CONFIGURATION_PROMPT = """
Generate a chart configuration based on this data: {data}
The goal is to show: {visualization_goal}
"""

CREATE_CHART_PROMPT = """
Write Python code to create a chart based on the configuration below.
Return only the code, no other text.
config: {config}
"""


class VisualizationConfig(BaseModel):
    """Configuration schema for chart generation."""
    chart_type: str = Field(..., description="Type of chart (e.g., bar, line, scatter).")
    x_axis: str = Field(..., description="Column name for the x-axis.")
    y_axis: str = Field(..., description="Column name for the y-axis.")
    title: str = Field(..., description="Chart title.")


@tracer.chain()
def extract_chart_config(data: str, visualization_goal: str) -> dict:
    """Generate a chart configuration. Emits a chain-typed span."""
    formatted_prompt = CHART_CONFIGURATION_PROMPT.format(
        data=data, visualization_goal=visualization_goal
    )
    # Structured Outputs. Verified against openai>=1.50 SDK, 2026-08-31.
    # See notebook 01 Part 1.3 for the API-version note.
    response = client.chat.completions.parse(
        model=MODEL,
        messages=[{"role": "user", "content": formatted_prompt}],
        response_format=VisualizationConfig,
    )
    try:
        content = response.choices[0].message.parsed
        return {
            "chart_type": content.chart_type,
            "x_axis": content.x_axis,
            "y_axis": content.y_axis,
            "title": content.title,
            "data": data
        }
    except Exception:
        return {"chart_type": "bar", "x_axis": "date", "y_axis": "value",
                "title": visualization_goal, "data": data}


@tracer.chain()
def create_chart(config: dict) -> str:
    """Generate chart code from a config dict. Emits a chain-typed span."""
    formatted_prompt = CREATE_CHART_PROMPT.format(config=config)
    response = client.chat.completions.create(
        model=MODEL,
        messages=[{"role": "user", "content": formatted_prompt}],
    )
    code = response.choices[0].message.content
    code = code.replace("```python", "").replace("```", "").strip()
    return code


@tracer.chain()
def generate_visualization(data: str, visualization_goal: str) -> str:
    """Generate visualization code. Emits a chain-typed span wrapping two child chains."""
    config = extract_chart_config(data, visualization_goal)
    code = create_chart(config)
    return code

In [ ]:
tools = [
    {
        "type": "function",
        "function": {
            "name": "lookup_sales_data",
            "description": "Look up sales transaction data from the Store Sales Price Elasticity Promotions dataset",
            "parameters": {
                "type": "object",
                "properties": {
                    "prompt": {"type": "string", "description": "The unchanged prompt that the user provided."}
                },
                "required": ["prompt"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "analyze_sales_data",
            "description": "Analyze sales data to extract business insights",
            "parameters": {
                "type": "object",
                "properties": {
                    "data": {"type": "string", "description": "The lookup_sales_data tool output."},
                    "prompt": {"type": "string", "description": "The unchanged prompt that the user provided."}
                },
                "required": ["data", "prompt"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "generate_visualization",
            "description": "Generate executable Python code to create a data visualization",
            "parameters": {
                "type": "object",
                "properties": {
                    "data": {"type": "string", "description": "The lookup_sales_data tool output."},
                    "visualization_goal": {"type": "string", "description": "Description of the chart to create."}
                },
                "required": ["data", "visualization_goal"]
            }
        }
    }
]

tool_implementations = {
    "lookup_sales_data": lookup_sales_data,
    "analyze_sales_data": analyze_sales_data,
    "generate_visualization": generate_visualization
}

SYSTEM_PROMPT = """
You are a helpful assistant that can answer questions about the Store Sales Price Elasticity Promotions dataset.
"""


@tracer.chain()
def handle_tool_calls(tool_calls, messages):
    """Execute tool calls and append results. Emits a chain-typed span."""
    for tool_call in tool_calls:
        function = tool_implementations[tool_call.function.name]
        function_args = json.loads(tool_call.function.arguments)
        result = function(**function_args)
        messages.append({
            "role": "tool",
            "content": result,
            "tool_call_id": tool_call.id
        })
    return messages


def run_agent(messages, system_prompt: str = SYSTEM_PROMPT):
    """Run the router loop until the LLM produces a final answer."""
    if isinstance(messages, str):
        messages = [{"role": "user", "content": messages}]

    if not any(isinstance(m, dict) and m.get("role") == "system" for m in messages):
        messages.insert(0, {"role": "system", "content": system_prompt})

    while True:
        response = client.chat.completions.create(
            model=MODEL,
            messages=messages,
            tools=tools,
        )
        messages.append(response.choices[0].message)
        tool_calls = response.choices[0].message.tool_calls

        if tool_calls:
            messages = handle_tool_calls(tool_calls, messages)
        else:
            return response.choices[0].message.content

<!--Navigate back to table of contents-->
<div align="left" style="text-align: left; background-color:#003262;">
    <span>
        <hr style="border: 8px solid#003262;" />
        <a style="color:#FFFFFF; background-color:#003262; border:1px solid #FFFFFF; border-color:#FFFFFF;border-radius:0px;border-width:0px;display:inline-block;font-family:arial,helvetica,sans-serif;font-size:24px;letter-spacing:0px;line-height:20px;padding:24px 40px;text-align:left;text-decoration:none; align:left"> 
            <strong>CONCEPT</strong> CHECK 
        </a>        
    </span>
</div>
<!-------------------------------------->

> **The `handle_tool_calls` function is decorated with `@tracer.chain()` but the `run_agent` loop is not decorated at all. The main agent span is created manually in `start_main_span`. Why is the agent span created with `tracer.start_as_current_span()` rather than a decorator, and what does the `openinference_span_kind="agent"` argument tell Phoenix?**

<br>

```python
# Write your explanation as comments.
```

<hr style="border: 2px solid#003262;" />

<!--Navigate back to table of contents-->
<div alig="right" style="text-align: right">
    <span>
        <a style="color:#FFFFFF; background-color:#003262; border:1px solid #FFFFFF; border-color:#FFFFFF;border-radius:5px;border-width:0px;display:inline-block;font-family:arial,helvetica,sans-serif;font-size:10px;letter-spacing:0px;line-height:10px;padding:10px 20px;text-align:center;text-decoration:none; align:center" href="#Part_table_contents" name="Table of Contents"  id="Part_table_contents"> 
            Table of Contents 
        </a>
    </span>
</div>
<!-------------------------------------->

<a id='Part_3'></a>

<hr style="border: 2px solid#003262;" />

#### PART 3

## **RUNNING A TRACED QUERY**: Observing the Trace

<div align="center" style="font-size:12px; font-family:FreeMono; font-weight: 100; font-stretch:ultra-condensed; line-height: 1.0; color:#2A2C2B">
    <img src="images/data_analysis_dark.png" align="center" width="30%" padding="10"><br>
    <br>
</div>

The `start_main_span` function wraps the entire agent run in a single agent-typed span. This is the root of the trace tree. Every LLM call, tool call, and chain step inside `run_agent` will appear as a child span nested under this root.

After running a query, open Phoenix at `http://localhost:6006` and navigate to the `tracing_agent` project. You should see one trace per `start_main_span` call. Click a trace to expand the span tree and inspect inputs, outputs, and latency at each step.

In [ ]:
def start_main_span(messages):
    """Wrap the entire agent run in a root agent-typed span."""
    with tracer.start_as_current_span(
        "AgentRun", openinference_span_kind="agent"
    ) as span:
        span.set_input(value=messages)
        result = run_agent(messages)
        span.set_output(value=result)
        span.set_status(StatusCode.OK)
        return result

In [ ]:
# Run a query through the fully traced agent.
# After this cell executes, inspect the trace in Phoenix at http://localhost:6006
user_question = "Show me the code for a scatterplot of sales by store in November 2021, and tell me what trends you see."
result = start_main_span([{"role": "user", "content": user_question}])
Markdown(result)

<br>

Once the query above finishes, the Phoenix UI at `http://localhost:6006` will show a tree of nested spans. The Concept Check below asks you to sketch what the tree should look like; the diagram below is the intended answer. Attempt your own sketch first, then compare.

<div align="center" style="font-size:12px; font-family:FreeMono; font-weight: 100; font-stretch:ultra-condensed; line-height: 1.0; color:#2A2C2B">
    <img src="images/span_tree.png" align="center" width="70%" padding="10"><br>
    <br>
    A canonical trace tree for a query that triggers all three tools. Colors correspond to span kinds (agent, chain, tool, LLM).
</div>

Two things to notice as you read the tree:

- Every leaf is an **LLM** span. That is auto-instrumentation at work - you did not write any code to emit those.
- Every branch is either a **chain** (linking steps in a pipeline) or a **tool** (a callable the router chose). Notice that `generate_visualization` is a chain wrapping two child chains, each of which wraps an LLM call. That nesting is exactly the "structure first, generate second" pattern from notebook 01.

<!--Navigate back to table of contents-->
<div align="left" style="text-align: left; background-color:#003262;">
    <span>
        <hr style="border: 8px solid#003262;" />
        <a style="color:#FFFFFF; background-color:#003262; border:1px solid #FFFFFF; border-color:#FFFFFF;border-radius:0px;border-width:0px;display:inline-block;font-family:arial,helvetica,sans-serif;font-size:24px;letter-spacing:0px;line-height:20px;padding:24px 40px;text-align:left;text-decoration:none; align:left"> 
            <strong>CONCEPT</strong> CHECK 
        </a>        
    </span>
</div>
<!-------------------------------------->

> **In Phoenix, a trace for this agent should show spans of types: LLM (auto-instrumented), tool (lookup and analysis), and chain (handle_tool_calls, visualization steps). Sketch the expected span tree for a query that triggers all three tools. Use indentation to show parent-child relationships.**

<br>

```python
# Sketch the span tree as comments.
# AgentRun (agent)
#   ...
#     ...
```

<hr style="border: 2px solid#003262;" />

<!--Navigate back to table of contents-->
<div alig="right" style="text-align: right">
    <span>
        <a style="color:#FFFFFF; background-color:#003262; border:1px solid #FFFFFF; border-color:#FFFFFF;border-radius:5px;border-width:0px;display:inline-block;font-family:arial,helvetica,sans-serif;font-size:10px;letter-spacing:0px;line-height:10px;padding:10px 20px;text-align:center;text-decoration:none; align:center" href="#Part_table_contents" name="Table of Contents"  id="Part_table_contents"> 
            Table of Contents 
        </a>
    </span>
</div>
<!-------------------------------------->

<a id='Part_4'></a>

<hr style="border: 2px solid#003262;" />

#### PART 4

## **EVALUATION-DRIVEN IMPROVEMENT**: From Trace to Fix

Tracing is only valuable if it changes what you do next. With traces landed in Phoenix, you can now inspect individual spans to find:

- **Which tool is called most often** - and whether it is the right one for the incoming query.
- **What the SQL span actually returned** - hallucinated columns, missing filters, and unformatted output are visible in the LLM span input/output pair.
- **How long each tool takes** - latency is recorded per span, which makes tail-latency outliers easy to spot.

The EDD improvement cycle, executed against this trace data, is:

1. **Run the agent** on a batch of representative queries so Phoenix has enough spans to observe.
2. **Inspect traces (or run evals against them)** to identify patterns in failures.
3. **Make one targeted change** - a prompt, a tool description, a parameter. Only one.
4. **Re-run and compare** - do the failure-mode spans disappear, improve, or stay the same?

Manual inspection of traces is the version of this loop that scales to tens of queries. Beyond that, you need automated scoring - which is exactly what the next notebook does. `03_router_and_skill_evals.ipynb` runs four evaluations against the spans this notebook produces (tool calling, code runnability, response clarity, SQL generation) and writes the scores back to Phoenix as annotations. The improved SQL prompt from notebook 01 is re-applied there as the demonstration fix, closing the loop end to end.

___

**Note:** Making one change at a time is deliberate. Changing the SQL prompt, the tool description, and the model simultaneously makes it impossible to attribute an improvement (or a regression) to any single change. In an eval-driven workflow, this discipline is what lets a rising score actually mean something.

___

<!--Navigate back to table of contents-->
<div align="left" style="text-align: left; background-color:#003262;">
    <span>
        <hr style="border: 8px solid#003262;" />
        <a style="color:#FFFFFF; background-color:#003262; border:1px solid #FFFFFF; border-color:#FFFFFF;border-radius:0px;border-width:0px;display:inline-block;font-family:arial,helvetica,sans-serif;font-size:24px;letter-spacing:0px;line-height:20px;padding:24px 40px;text-align:left;text-decoration:none; align:left"> 
            <strong>CONCEPT</strong> CHECK 
        </a>        
    </span>
</div>
<!-------------------------------------->

> **Tracing tells you what happened but not whether it was correct. Name the three types of evaluations used in notebook 03 (tool calling, code runnability, response clarity) and write one sentence explaining what each one measures that a trace alone cannot tell you.**

<br>

```python
# Write your explanation as comments:
# Tool calling eval measures...
# Code runnability eval measures...
# Response clarity eval measures...
```

<hr style="border: 2px solid#003262;" />

___

**Next:** `03_router_and_skill_evals.ipynb` runs LLM-as-judge evaluations across a batch of questions, scores tool selection correctness, code runnability, and response clarity, then writes the scores back to Phoenix so they appear alongside the traces.

___

<hr style="border: 6px solid#003262;" />